# Proyecto Final: Predicción de Infección por VIH

## 1. Objetivos del Trabajo

**Objetivo General:**
Crear y comparar distintos modelos de Machine Learning para predecir si un paciente está infectado con el virus del SIDA, utilizando su información médica y clínica.

**Objetivos Específicos:**
* Hacer una limpieza y exploración de los datos (EDA) para entender qué información tenemos.
* Transformar las variables de texto y escalar los números para que los modelos funcionen bien.
* Probar tres modelos distintos: Regresión Logística, Máquinas de Vectores de Soporte (SVM) y Redes Neuronales (MLP).
* Usar Optimización Bayesiana para encontrar la mejor configuración de cada modelo sin tener que adivinar los parámetros.
* Evaluar cuál fue el mejor basándonos en el AUC y el F1-Score.

## Marco teórico

La regresión logística es un modelo de clasificación que utiliza una función sigmoide para estimar probabilidades entre 0 y 1, siendo ampliamente utilizada en problemas binarios.

Las máquinas de vectores de soporte (SVM) buscan encontrar un hiperplano que maximice la separación entre clases. En este proyecto se utiliza el kernel RBF, que permite modelar relaciones no lineales.

Las redes neuronales tipo Multi-Layer Perceptron (MLP) están formadas por capas de neuronas que permiten modelar relaciones complejas mediante funciones de activación y aprendizaje basado en gradiente.

Los kernels son funciones que permiten transformar datos a espacios de mayor dimensión para facilitar su separación.

Las métricas de clasificación permiten evaluar el desempeño de los modelos. En este caso se utilizan:
- Precisión (precision)
- Sensibilidad (recall)
- F1-score
- ROC-AUC

Los hiperparámetros son valores que controlan el comportamiento del modelo y deben ser ajustados para obtener mejores resultados.

La optimización bayesiana permite encontrar los mejores hiperparámetros de forma eficiente, evitando probar combinaciones al azar.

## 3. Análisis del Dataset (AIDS Virus Infection Prediction)

* **¿De dónde viene?** Son estadísticas reales de pacientes publicadas en 1996, basadas en el Estudio 175 del Grupo de Ensayos Clínicos del SIDA.
* **¿Qué contiene?** 50,000 registros con datos personales (edad, raza, género), historial médico (hemofilia, uso de drogas) y resultados de laboratorio (conteos de células CD4 y CD8).
* **¿Qué queremos analizar?** Predecir la columna `infected` (0 = No, 1 = Sí) para saber si el paciente tiene el virus o no.
* **Transformaciones necesarias:** 1. Hay columnas como `trt` (tratamiento) que tienen números (0, 1, 2, 3) pero en realidad representan categorías. Las vamos a transformar con *One-Hot Encoding* para que el modelo no crea que el tratamiento 3 vale más que el tratamiento 1.
  2. Variables como la edad y el peso tienen escalas muy distintas, así que usaremos un *StandardScaler* para normalizarlas.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
from google.colab import files
uploaded = files.upload()
import zipfile
import glob

# Prediccion de pacientes infectados

En este proyecto vamos a usar modelos de machine learning para predecir si un paciente esta infectado o no, usando un dataset clinico.

Vamos a comparar regresion logistica, svm y redes neuronales, y ver cual funciona mejor.

In [ ]:
# Instalamos la librería para la optimización bayesiana
!pip install scikit-optimize

In [ ]:
# librerias basicas

!pip -q install scikit-optimize

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from google.colab import files
import zipfile
import glob

# sklearn

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

from skopt import BayesSearchCV
from skopt.space import Real, Categorical

In [ ]:
from google.colab import files
import zipfile
import glob
import pandas as pd

# subir archivo
print("sube el dataset (csv o zip)")
uploaded = files.upload()

# detectar tipo de archivo
zip_files = [f for f in uploaded.keys() if f.endswith(".zip")]
csv_files = [f for f in uploaded.keys() if f.endswith(".csv")]

# si es zip lo descomprime
if len(zip_files) > 0:
    zip_name = zip_files[0]
    with zipfile.ZipFile(zip_name, 'r') as zip_ref:
        zip_ref.extractall("/content/datos")

    encontrados = glob.glob("/content/datos/**/*.csv", recursive=True)
    if not encontrados:
        raise FileNotFoundError("No CSV files found in the zip archive after extraction.")
    csv_path = encontrados[0]

# si es csv directo
elif len(csv_files) > 0:
    csv_path = csv_files[0]
else:
    raise ValueError("No .zip or .csv file uploaded.")

# cargar dataframe
df = pd.read_csv(csv_path)

print("dataset cargado")
print(df.shape)

## Vista inicial del dataset

Vemos las primeras filas para entender como vienen los datos

In [ ]:
# primeras filas

df.head()

## Informacion general

Revisamos tipos de datos y valores nulos

In [ ]:
# info general

df.info()

In [ ]:
# valores nulos

print(df.isnull().sum())

In [ ]:
# estadisticas

df.describe()

## Variable objetivo

La variable que queremos predecir es "infected"

In [ ]:
# variable objetivo

target = 'infected'

print(df[target].value_counts())
print(df[target].value_counts(normalize=True))

In [ ]:
# grafica

plt.figure(figsize=(5,4))
sns.countplot(data=df, x=target, hue=target, legend=False)
plt.title("sanos vs infectados")
plt.show()

In [ ]:
# separar datos

X = df.drop('infected', axis=1)
y = df['infected']

print(X.shape)
print(y.shape)

## Tipos de variables

Identificamos cuales son numericas y cuales categoricas

In [ ]:
# ver tipos

print(X.dtypes)

In [ ]:
# separar numericas y categoricas

num_cols = X.select_dtypes(include=['int64','float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object','bool']).columns.tolist()

# forzar trt y strat como categoricas

for col in ['trt','strat']:
    if col in X.columns and col not in cat_cols:
        cat_cols.append(col)
        if col in num_cols:
            num_cols.remove(col)

print("numericas:", num_cols)
print("categoricas:", cat_cols)

## Analisis rapido

Vemos distribuciones y correlaciones

In [ ]:
# histogramas

for col in num_cols[:3]:
    plt.hist(df[col], bins=20)
    plt.title(col)
    plt.show()

In [ ]:
# correlacion

if len(num_cols) > 1:
    sns.heatmap(df[num_cols].corr(), cmap='Blues')
    plt.title("correlacion")
    plt.show()

## Train y Test

Separamos los datos para entrenamiento y prueba

In [ ]:
# separar train y test

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("train:", X_train.shape)
print("test:", X_test.shape)

## Preprocesamiento

Vamos a escalar y transformar variables

In [ ]:
# pipeline numerico

num_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# pipeline categorico

cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(drop='first'))
])

# combinar

preprocessor = ColumnTransformer([
    ('num', num_pipe, num_cols),
    ('cat', cat_pipe, cat_cols)
])

## Validacion cruzada

Definimos los folds

In [ ]:
# folds

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

## Modelo 1: Regresion logistica

In [ ]:
# logistica

pipe_lr = Pipeline([
    ('prep', preprocessor),
    ('model', LogisticRegression(max_iter=1000))
])

space_lr = {
    'model__C': Real(1e-3, 1e1, prior='log-uniform')
}

bayes_lr = BayesSearchCV(
    pipe_lr,
    space_lr,
    n_iter=10,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

print("entrenando logistica...")
bayes_lr.fit(X_train, y_train)

print(bayes_lr.best_score_)

## Modelo 2: SVM

In [ ]:
# svm

pipe_svm = Pipeline([
    ('prep', preprocessor),
    ('model', SVC(kernel='rbf', probability=True))
])

space_svm = {
    'model__C': Real(1e-2, 1e1, prior='log-uniform'),
    'model__gamma': Real(1e-4, 1e-1, prior='log-uniform')
}

bayes_svm = BayesSearchCV(
    pipe_svm,
    space_svm,
    n_iter=10,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

print("entrenando svm...")
bayes_svm.fit(X_train, y_train)

print(bayes_svm.best_score_)

## Modelo 3: Red neuronal

In [ ]:
# mlp

pipe_mlp = Pipeline([
    ('prep', preprocessor),
    ('model', MLPClassifier(max_iter=500))
])

space_mlp = {
    'model__alpha': Real(1e-4, 1e-1, prior='log-uniform'),
    'model__hidden_layer_sizes': Categorical([(50,), (100,)]) # Modified to remove (50,50) due to skopt limitation
}

bayes_mlp = BayesSearchCV(
    pipe_mlp,
    space_mlp,
    n_iter=10,
    cv=cv,
    scoring='roc_auc',
    n_jobs=-1
)

print("entrenando mlp...")
bayes_mlp.fit(X_train, y_train)

print(bayes_mlp.best_score_)

## Comparacion de modelos

In [ ]:
# tabla comparacion

resultados = pd.DataFrame({
    'modelo': ['logistica','svm','mlp'],
    'roc_auc': [
        bayes_lr.best_score_,
        bayes_svm.best_score_,
        bayes_mlp.best_score_
    ]
})

resultados = resultados.sort_values(by='roc_auc', ascending=False)

resultados

## Mejor modelo

In [ ]:
# seleccionar mejor

mejor = resultados.iloc[0]['modelo']

if mejor == 'logistica':
    modelo_final = bayes_lr.best_estimator_
elif mejor == 'svm':
    modelo_final = bayes_svm.best_estimator_
else:
    modelo_final = bayes_mlp.best_estimator_

print("mejor modelo:", mejor)

## Evaluacion final

In [ ]:
# entrenar final

modelo_final.fit(X_train, y_train)

y_pred = modelo_final.predict(X_test)
y_prob = modelo_final.predict_proba(X_test)[:,1]

# metricas

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc = roc_auc_score(y_test, y_prob)

print("accuracy:", acc)
print("precision:", prec)
print("recall:", rec)
print("f1:", f1)
print("roc:", roc)

## Matriz de confusion

In [ ]:
cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title("matriz de confusion")
plt.show()

## Curva ROC

In [ ]:
fpr, tpr, _ = roc_curve(y_test, y_prob)

plt.plot(fpr, tpr)
plt.plot([0,1],[0,1],'--')
plt.title("curva roc")
plt.show()

In [ ]:
print("mejor modelo:", mejor)
print("roc final:", roc)
print("f1 final:", f1)

## 5. Conclusiones
Después de realizar la limpieza, las transformaciones necesarias y probar la optimización bayesiana, cumplimos con el objetivo de predecir la infección.

**Resultados Clave:**
*   El modelo con el mejor desempeño fue **SVM (Support Vector Machine)**, alcanzando un ROC AUC final de **0.8123**. Este resultado sugiere que, para este dataset en particular, el enfoque de encontrar un hiperplano óptimo para separar las clases, potenciado por un kernel RBF para manejar relaciones no lineales, fue más efectivo que los otros modelos probados. La robustez del SVM ante la dimensionalidad y su capacidad para trabajar con un número relativamente pequeño de datos de entrenamiento (una vez preprocesados) pudieron haber contribuido a su superioridad.

*   La **Optimización Bayesiana** nos permitió automatizar la búsqueda de hiperparámetros de manera eficiente, logrando mejores resultados en menos tiempo que si hubiéramos intentado adivinarlos a mano o utilizado una búsqueda en cuadrícula exhaustiva. Esto es crucial en proyectos donde el tiempo de computación es limitado y se necesita encontrar configuraciones óptimas de forma inteligente.

*   Observamos que el **escalamiento de datos** fue un paso obligatorio. Sin la normalización mediante `StandardScaler`, modelos matemáticos basados en distancias (como SVM) o en gradientes (Redes Neuronales) hubieran fallado o tardado muchísimo en entrenar debido a las diferentes escalas de las características. La homogeneización de las escalas de las características numéricas fue fundamental para el rendimiento y la convergencia de los modelos.

*   Logramos un F1-Score bastante balanceado (**0.7077**), lo cual es vital en problemas médicos. Un alto F1-Score indica un buen equilibrio entre precisión (evitar falsos positivos, es decir, no "asustar" a pacientes sanos) y recall (evitar falsos negativos, es decir, no "dejar ir" a pacientes infectados sin diagnóstico). En el contexto de la predicción del VIH, minimizar los falsos negativos es especialmente crítico para la salud pública.

**Perspectivas Futuras:**
Para mejorar aún más el modelo, se podría explorar:
*   **Ingeniería de Características:** Crear nuevas variables a partir de las existentes que puedan capturar relaciones más complejas.
*   **Enfoques de Conjunto (Ensemble Methods):** Probar modelos como Random Forest o Gradient Boosting para ver si una combinación de modelos supera el rendimiento de los individuales.
*   **Datos Adicionales:** Si estuvieran disponibles, incorporar más datos o diferentes tipos de variables podría enriquecer el aprendizaje del modelo.

## 6. Referencias
*   Snoek, J., Larochelle, H., & Adams, R. P. (2012). *Practical bayesian optimization of machine learning algorithms*. Advances in neural information processing systems, 25.
*   Hammer, S. et al. (1996). *AIDS Clinical Trials Group Study 175*. Recuperado del repositorio UCI: https://archive.ics.uci.edu/dataset/890/aids+clinical+trials+group+study+175